In [1]:
from collections import deque
import torch
from torch import nn, optim
from torch.distributions import Normal
from torch.nn.functional import mse_loss
from tqdm import trange

In [2]:
state_dim = 4
message_dim = 32
action_dim = state_dim
hidden_dim = 64
num_layers = 3

def reward_fn(state, action):
    return 100 - ((state - action) ** 2).sum(dim=-1, keepdim=True)

In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
        super(MLP, self).__init__()
        layers = []
        for i in range(num_layers):
            layers.append(nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.LeakyReLU())
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, *inputs):
        return self.net(torch.cat(inputs, dim=-1))
    
    def get_normal_policy(self, *inputs):
        mean, log_std = self.forward(*inputs).chunk(2, dim=-1)
        std = log_std.clamp(-20, 2).exp()
        return Normal(mean, std)

In [4]:
message1_actor, target_message1_actor = (MLP(state_dim, 2 * message_dim, hidden_dim, num_layers) for _ in range(2))
message2_actor, target_message2_actor = (MLP(message_dim, 2 * message_dim, hidden_dim, num_layers) for _ in range(2))
final_actor, target_final_actor = (MLP(message_dim, 2 * action_dim, hidden_dim, num_layers) for _ in range(2))

message1_critic, target_message1_critic = (MLP(state_dim, 1, hidden_dim, num_layers) for _ in range(2))
message2_critic, target_message2_critic = (MLP(state_dim + message_dim, 1, hidden_dim, num_layers) for _ in range(2))
final_critic, target_final_critic = (MLP(state_dim + 2 * message_dim, 1, hidden_dim, num_layers) for _ in range(2))

online_models = [message1_actor, message2_actor, final_actor, message1_critic, message2_critic, final_critic]
target_models = [target_message1_actor, target_message2_actor, target_final_actor, target_message1_critic, target_message2_critic, target_final_critic]

for target, online in zip(target_models, online_models):
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)

message1_actor_opt = optim.Adam(message1_actor.parameters(), lr=3e-4)
message2_actor_opt = optim.Adam(message2_actor.parameters(), lr=3e-4)
final_actor_opt = optim.Adam(final_actor.parameters(), lr=3e-4)
message1_critic_opt = optim.Adam(message1_critic.parameters(), lr=3e-4)
message2_critic_opt = optim.Adam(message2_critic.parameters(), lr=3e-4)
final_critic_opt = optim.Adam(final_critic.parameters(), lr=3e-4)

In [5]:
replay_capacity = 10000
num_episodes = 50000
batch_size = 64
epsilon = 0.1
k = 4
lmbda1 = 2.0
lmbda2 = 1.0
alpha = 0.1
tau = 0.005

In [6]:
# message_rollout_buffer = deque(maxlen=10000)
# final_rollout_buffer = deque(maxlen=10000)
# This isn't needed right now, since the routing is fixed for every sampling.
# We can just rely on using a batch, instead of a replay buffer, to collect samples.
def polyak_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

In [7]:
def weighted_mae(pred, target, lmbda1, lmbda2):
    return lmbda1 * torch.relu(pred - target) + lmbda2 * torch.relu(target - pred)

In [ ]:
for episode in trange(num_episodes):
    # Collect rollout samples
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)
        message1_policy = message1_actor.get_normal_policy(state)
        message1 = message1_policy.sample()
        message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)
        message2_value = message2_critic(state, message1)
        message2_policy = message2_actor.get_normal_policy(message1)
        message2 = message2_policy.sample()
        message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)
        final_value = final_critic(state, message1, message2)
        final_policy = final_actor.get_normal_policy(message2)
        action = final_policy.sample()
        action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)
        final_reward = reward_fn(state, action)
        final_error = weighted_mae(final_value, final_reward, lmbda1, lmbda2)
        message2_reward = final_reward #final_value - final_error
        message2_error = weighted_mae(message2_value, message2_reward, lmbda1, lmbda2)
        message1_reward = final_reward #message2_value - message2_error

    # Update critics
    for i in range(k):
        mb_state = state[i * batch_size:(i + 1) * batch_size]
        mb_message1 = message1[i * batch_size:(i + 1) * batch_size]
        mb_message1_log_prob = message1_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_message2 = message2[i * batch_size:(i + 1) * batch_size]
        mb_message2_log_prob = message2_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action = action[i * batch_size:(i + 1) * batch_size]
        mb_action_log_prob = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward = final_reward[i * batch_size:(i + 1) * batch_size]
        mb_message2_reward = message2_reward[i * batch_size:(i + 1) * batch_size]
        mb_message1_reward = message1_reward[i * batch_size:(i + 1) * batch_size]

        # Update message critic
        message1_critic_opt.zero_grad()
        message1_critic_value = message1_critic(mb_state)
        message1_critic_loss = mse_loss(message1_critic_value, mb_message1_reward)
        message1_critic_loss.backward()
        message1_critic_opt.step()

        message2_critic_opt.zero_grad()
        message2_critic_value = message2_critic(mb_state, mb_message1)
        message2_critic_loss = mse_loss(message2_critic_value, mb_message2_reward)
        message2_critic_loss.backward()
        message2_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_value = final_critic(mb_state, mb_message1, mb_message2)
        final_critic_loss = mse_loss(final_critic_value, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # Update actors
        if episode % 3 == 0:
            message1_actor_opt.zero_grad()
            message1_policy = message1_actor.get_normal_policy(mb_state)
            message1_new_log_prob = message1_policy.log_prob(mb_message1).sum(dim=-1, keepdim=True)
            message1_prob_ratio = (message1_new_log_prob - mb_message1_log_prob).exp()
            message1_clipped_prob_ratio = message1_prob_ratio.clamp(1 - epsilon, 1 + epsilon)
            message1_advantage = mb_message1_reward - message1_critic_value.detach()
            message1_advantage = (message1_advantage - message1_advantage.mean()) / (message1_advantage.std() + 1e-8)
            message1_actor_loss1 = message1_prob_ratio * message1_advantage
            message1_actor_loss2 = message1_clipped_prob_ratio * message1_advantage
            message1_policy_ent = message1_policy.entropy().mean()
            message1_actor_loss = -torch.min(message1_actor_loss1, message1_actor_loss2).mean() - alpha * message1_policy_ent
            message1_actor_loss.backward()
            message1_actor_opt.step()

        elif episode % 3 == 1:
            message2_actor_opt.zero_grad()
            message2_policy = message2_actor.get_normal_policy(mb_message1)
            message2_new_log_prob = message2_policy.log_prob(mb_message2).sum(dim=-1, keepdim=True)
            message2_prob_ratio = (message2_new_log_prob - mb_message2_log_prob).exp()
            message2_clipped_prob_ratio = message2_prob_ratio.clamp(1 - epsilon, 1 + epsilon)
            message2_advantage = mb_message2_reward - message2_critic_value.detach()
            message2_advantage = (message2_advantage - message2_advantage.mean()) / (message2_advantage.std() + 1e-8)
            message2_actor_loss1 = message2_prob_ratio * message2_advantage
            message2_actor_loss2 = message2_clipped_prob_ratio * message2_advantage
            message2_policy_ent = message2_policy.entropy().mean()
            message2_actor_loss = -torch.min(message2_actor_loss1, message2_actor_loss2).mean() - alpha * message2_policy_ent
            message2_actor_loss.backward()
            message2_actor_opt.step()

        else:
            final_actor_opt.zero_grad()
            final_policy = final_actor.get_normal_policy(mb_message2)
            final_new_log_prob = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
            final_prob_ratio = (final_new_log_prob - mb_action_log_prob).exp()
            final_clipped_prob_ratio = final_prob_ratio.clamp(1 - epsilon, 1 + epsilon)
            final_advantage = mb_final_reward - final_critic_value.detach()
            final_advantage = (final_advantage - final_advantage.mean()) / (final_advantage.std() + 1e-8)
            final_actor_loss1 = final_prob_ratio * final_advantage
            final_actor_loss2 = final_clipped_prob_ratio * final_advantage
            final_policy_ent = final_policy.entropy().mean()
            final_actor_loss = -torch.min(final_actor_loss1, final_actor_loss2).mean() - alpha * final_policy_ent
            final_actor_loss.backward()
            final_actor_opt.step()

    if episode % 100 == 0:
        print(f"{episode}, fcl: {final_critic_loss.item():.4f}, m1cl: {message1_critic_loss.item():.4f}, m2cl: {message2_critic_loss.item():.4f}, fr: {final_reward.mean().item():.4f}, m1r: {message1_reward.mean().item():.4f}, m2r: {message2_reward.mean().item():.4f}, fv: {final_value.mean().item():.4f}, m1v: {message1_critic_value.mean().item():.4f}, m2v: {message2_critic_value.mean().item():.4f}, fe: {final_error.mean().item():.4f}, m2e: {message2_policy_ent.item():.4f}")

In [ ]:
with torch.no_grad():
    state = torch.randn(k * batch_size, state_dim)
    message1_policy = message1_actor.get_normal_policy(state)
    message1 = message1_policy.sample()
    message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)
    message2_value = message2_critic(state, message1)
    message2_policy = message2_actor.get_normal_policy(message1)
    message2 = message2_policy.sample()
    #message2 = message2.flip(dims=[0])
    message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)
    final_value = final_critic(state, message1, message2)
    final_policy = final_actor.get_normal_policy(message2)
    action = final_policy.sample()
    action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)
    final_reward = reward_fn(state, action)
    final_error = weighted_mae(final_value, final_reward, lmbda1, lmbda2)
    message2_reward = final_value - final_error
    message2_error = weighted_mae(message2_value, message2_reward, lmbda1, lmbda2)
    message1_reward = message2_reward - message2_error
    print(f"Final Reward: {final_reward.mean().item():.4f}, Final Value: {final_value.mean().item():.4f}, Message1 Reward: {message1_reward.mean().item():.4f}, Message2 Reward: {message2_reward.mean().item():.4f}")

Final Reward: 96.1139, Final Value: 96.1333, Message1 Reward: 95.9188, Message2 Reward: 96.0478


In [18]:
message1_actor = MLP(state_dim, 2 * message_dim, hidden_dim, num_layers)
message2_actor = MLP(state_dim + message_dim, 2 * message_dim, hidden_dim, num_layers)
final_actor    = MLP(message_dim, 2 * action_dim, hidden_dim, num_layers)

message1_critic = MLP(state_dim, 1, hidden_dim, num_layers)
message2_critic = MLP(state_dim + message_dim, 1, hidden_dim, num_layers)
final_critic    = MLP(state_dim + 2 * message_dim, 1, hidden_dim, num_layers)

message1_actor_opt  = optim.Adam(message1_actor.parameters(),  lr=3e-4)
message2_actor_opt  = optim.Adam(message2_actor.parameters(),  lr=3e-4)
final_actor_opt     = optim.Adam(final_actor.parameters(),     lr=3e-4)
message1_critic_opt = optim.Adam(message1_critic.parameters(), lr=3e-4)
message2_critic_opt = optim.Adam(message2_critic.parameters(), lr=3e-4)
final_critic_opt    = optim.Adam(final_critic.parameters(),    lr=3e-4)

In [20]:
# I AM THE LIVING MAN
for episode in trange(num_episodes):
    # Collect rollout samples
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)

        message1_policy = message1_actor.get_normal_policy(state)
        message1 = message1_policy.sample()
        message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)

        message2_policy = message2_actor.get_normal_policy(torch.zeros_like(state), message1)
        message2 = message2_policy.sample()
        message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)

        final_policy = final_actor.get_normal_policy(message2)
        action = final_policy.sample()
        action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)

        final_reward = reward_fn(state, action)

    # Update critics and actors
    for i in range(k):
        mb_state           = state[i * batch_size:(i + 1) * batch_size]
        mb_message1        = message1[i * batch_size:(i + 1) * batch_size]
        mb_message1_lp     = message1_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_message2        = message2[i * batch_size:(i + 1) * batch_size]
        mb_message2_lp     = message2_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action          = action[i * batch_size:(i + 1) * batch_size]
        mb_action_lp       = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward    = final_reward[i * batch_size:(i + 1) * batch_size]

        # ── Critics ──────────────────────────────────────────────────────────
        message1_critic_opt.zero_grad()
        message1_critic_value = message1_critic(mb_state)                        # V(s)
        message2_critic_value = message2_critic(mb_state, mb_message1)           # V(s, m1)
        final_critic_value    = final_critic(mb_state, mb_message1, mb_message2) # V(s, m1, m2)

        message1_critic_loss = mse_loss(message1_critic_value, message2_critic_value.detach())
        message1_critic_loss.backward()
        message1_critic_opt.step()

        message2_critic_opt.zero_grad()
        message2_critic_loss = mse_loss(message2_critic_value, final_critic_value.detach())
        message2_critic_loss.backward()
        message2_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_loss = mse_loss(final_critic_value, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # ── Marginal advantages ───────────────────────────────────────────────
        # Recompute with fresh critic values (post-update)
        with torch.no_grad():
            v0 = message1_critic(mb_state)
            v1 = message2_critic(mb_state, mb_message1)
            v2 = final_critic(mb_state, mb_message1, mb_message2)

        message1_advantage = v1 - v0                        # what did m1 add
        message2_advantage = v2 - v1                        # what did m2 add
        final_advantage    = mb_final_reward - v2           # what did action add

        def normalize(adv):
            return (adv - adv.mean()) / (adv.std() + 1e-8)

        message1_advantage = normalize(message1_advantage)
        message2_advantage = normalize(message2_advantage)
        final_advantage    = normalize(final_advantage)

        # ── Actor 1 ───────────────────────────────────────────────────────────
        message1_actor_opt.zero_grad()
        message1_policy       = message1_actor.get_normal_policy(mb_state)
        message1_new_lp       = message1_policy.log_prob(mb_message1).sum(dim=-1, keepdim=True)
        message1_ratio        = (message1_new_lp - mb_message1_lp).exp()
        message1_ratio_clip   = message1_ratio.clamp(1 - epsilon, 1 + epsilon)
        message1_ent          = message1_policy.entropy().mean()
        message1_actor_loss   = -torch.min(
            message1_ratio * message1_advantage,
            message1_ratio_clip * message1_advantage
        ).mean() - alpha * message1_ent
        message1_actor_loss.backward()
        message1_actor_opt.step()

        # ── Actor 2 ───────────────────────────────────────────────────────────
        message2_actor_opt.zero_grad()
        message2_policy       = message2_actor.get_normal_policy(torch.zeros_like(mb_state), mb_message1)
        message2_new_lp       = message2_policy.log_prob(mb_message2).sum(dim=-1, keepdim=True)
        message2_ratio        = (message2_new_lp - mb_message2_lp).exp()
        message2_ratio_clip   = message2_ratio.clamp(1 - epsilon, 1 + epsilon)
        message2_ent          = message2_policy.entropy().mean()
        message2_actor_loss   = -torch.min(
            message2_ratio * message2_advantage,
            message2_ratio_clip * message2_advantage
        ).mean() - alpha * message2_ent
        message2_actor_loss.backward()
        message2_actor_opt.step()

        # ── Final actor ───────────────────────────────────────────────────────
        final_actor_opt.zero_grad()
        final_policy          = final_actor.get_normal_policy(mb_message2)
        final_new_lp          = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
        final_ratio           = (final_new_lp - mb_action_lp).exp()
        final_ratio_clip      = final_ratio.clamp(1 - epsilon, 1 + epsilon)
        final_ent             = final_policy.entropy().mean()
        final_actor_loss      = -torch.min(
            final_ratio * final_advantage,
            final_ratio_clip * final_advantage
        ).mean() - alpha * final_ent
        final_actor_loss.backward()
        final_actor_opt.step()

    if episode % 100 == 0:
        print(
            f"{episode} | "
            f"fcl: {final_critic_loss.item():.4f} | "
            f"m2cl: {message2_critic_loss.item():.4f} | "
            f"m1cl: {message1_critic_loss.item():.4f} | "
            f"fr: {final_reward.mean().item():.4f}"
        )

  0%|          | 0/50000 [00:00<?, ?it/s]

  0%|          | 2/50000 [00:00<1:39:18,  8.39it/s]

0 | fcl: 0.0394 | m2cl: 0.0359 | m1cl: 0.0094 | fr: 99.6522


  0%|          | 103/50000 [00:09<1:14:31, 11.16it/s]

100 | fcl: 0.0207 | m2cl: 0.0547 | m1cl: 0.0087 | fr: 99.7162


  0%|          | 150/50000 [00:13<1:17:22, 10.74it/s]


KeyboardInterrupt: 

In [25]:
# Actors — decentralized, only see what flows through the chain
message1_actor = MLP(state_dim,              2 * message_dim, hidden_dim, num_layers)
message2_actor = MLP(message_dim,            2 * message_dim, hidden_dim, num_layers)
final_actor    = MLP(message_dim,            2 * action_dim,  hidden_dim, num_layers)

# Critics — centralized, see state during training
message1_critic = MLP(state_dim,                          1, hidden_dim, num_layers)
message2_critic = MLP(state_dim + message_dim,            1, hidden_dim, num_layers)
final_critic    = MLP(state_dim + 2 * message_dim,        1, hidden_dim, num_layers)

for episode in trange(num_episodes):
    # ── Rollout ───────────────────────────────────────────────────────────────
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)

        # Actors only see what flows through the chain (decentralized execution)
        message1_policy   = message1_actor.get_normal_policy(state)
        message1          = message1_policy.sample()
        message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)

        message2_policy   = message2_actor.get_normal_policy(message1)
        message2          = message2_policy.sample()
        message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)

        final_policy      = final_actor.get_normal_policy(message2)
        action            = final_policy.sample()
        action_log_prob   = final_policy.log_prob(action).sum(dim=-1, keepdim=True)

        final_reward      = reward_fn(state, action)

    # ── Mini-batch updates ────────────────────────────────────────────────────
    for i in range(k):
        mb_state        = state[i * batch_size:(i + 1) * batch_size]
        mb_message1     = message1[i * batch_size:(i + 1) * batch_size]
        mb_message1_lp  = message1_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_message2     = message2[i * batch_size:(i + 1) * batch_size]
        mb_message2_lp  = message2_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action       = action[i * batch_size:(i + 1) * batch_size]
        mb_action_lp    = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward = final_reward[i * batch_size:(i + 1) * batch_size]

        # ── Critics (centralized — see state during training) ─────────────────
        # All critics predict E[final_reward | state, messages so far]
        # V(s), V(s, m1), V(s, m1, m2) form a telescoping estimate

        message1_critic_opt.zero_grad()
        message1_critic_value = message1_critic(mb_state)
        message1_critic_loss  = mse_loss(message1_critic_value, mb_final_reward)
        message1_critic_loss.backward()
        message1_critic_opt.step()

        message2_critic_opt.zero_grad()
        message2_critic_value = message2_critic(mb_state, mb_message1)
        message2_critic_loss  = mse_loss(message2_critic_value, mb_final_reward)
        message2_critic_loss.backward()
        message2_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_value = final_critic(mb_state, mb_message1, mb_message2)
        final_critic_loss  = mse_loss(final_critic_value, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # ── Marginal advantages (fresh critic values post-update) ─────────────
        with torch.no_grad():
            v0 = message1_critic(mb_state)
            v1 = message2_critic(mb_state, mb_message1)
            v2 = final_critic(mb_state, mb_message1, mb_message2)

        def normalize(adv):
            return (adv - adv.mean()) / (adv.std() + 1e-8)

        # Each agent gets credit only for what its message added
        message1_advantage = normalize(v1 - v0)
        message2_advantage = normalize(v2 - v1)
        final_advantage = normalize(mb_final_reward - v2)

        # ── Actor 1 (sees state only — first in chain) ────────────────────────
        message1_actor_opt.zero_grad()
        message1_policy     = message1_actor.get_normal_policy(mb_state)
        message1_new_lp     = message1_policy.log_prob(mb_message1).sum(dim=-1, keepdim=True)
        message1_ratio      = (message1_new_lp - mb_message1_lp).exp()
        message1_ratio_clip = message1_ratio.clamp(1 - epsilon, 1 + epsilon)
        message1_ent        = message1_policy.entropy().mean()
        message1_actor_loss = -torch.min(
            message1_ratio      * message1_advantage,
            message1_ratio_clip * message1_advantage
        ).mean() - alpha * message1_ent
        message1_actor_loss.backward()
        message1_actor_opt.step()

        # ── Actor 2 (sees message1 only — decentralized execution) ────────────
        message2_actor_opt.zero_grad()
        message2_policy     = message2_actor.get_normal_policy(mb_message1)
        message2_new_lp     = message2_policy.log_prob(mb_message2).sum(dim=-1, keepdim=True)
        message2_ratio      = (message2_new_lp - mb_message2_lp).exp()
        message2_ratio_clip = message2_ratio.clamp(1 - epsilon, 1 + epsilon)
        message2_ent        = message2_policy.entropy().mean()
        message2_actor_loss = -torch.min(
            message2_ratio      * message2_advantage,
            message2_ratio_clip * message2_advantage
        ).mean() - alpha * message2_ent
        message2_actor_loss.backward()
        message2_actor_opt.step()

        # ── Final actor (sees message2 only — decentralized execution) ────────
        final_actor_opt.zero_grad()
        final_policy        = final_actor.get_normal_policy(mb_message2)
        final_new_lp        = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
        final_ratio         = (final_new_lp - mb_action_lp).exp()
        final_ratio_clip    = final_ratio.clamp(1 - epsilon, 1 + epsilon)
        final_ent           = final_policy.entropy().mean()
        final_actor_loss    = -torch.min(
            final_ratio      * final_advantage,
            final_ratio_clip * final_advantage
        ).mean() - alpha * final_ent
        final_actor_loss.backward()
        final_actor_opt.step()

    if episode % 100 == 0:
        print(
            f"{episode} | "
            f"fcl: {final_critic_loss.item():.4f} | "
            f"m2cl: {message2_critic_loss.item():.4f} | "
            f"m1cl: {message1_critic_loss.item():.4f} | "
            f"fr: {final_reward.mean().item():.4f}"
        )

  0%|          | 3/50000 [00:00<1:15:15, 11.07it/s]

0 | fcl: 8379.0703 | m2cl: 8303.5049 | m1cl: 8370.2793 | fr: 90.6351


  0%|          | 103/50000 [00:08<1:03:43, 13.05it/s]

100 | fcl: 8168.0127 | m2cl: 8106.1914 | m1cl: 8162.7222 | fr: 90.6825


  0%|          | 203/50000 [00:16<1:09:06, 12.01it/s]

200 | fcl: 8165.9673 | m2cl: 8109.3135 | m1cl: 8160.7236 | fr: 91.2426


  1%|          | 303/50000 [00:24<1:05:33, 12.63it/s]

300 | fcl: 8238.7822 | m2cl: 8187.7256 | m1cl: 8247.1094 | fr: 90.8668


  1%|          | 349/50000 [00:28<1:06:34, 12.43it/s]


KeyboardInterrupt: 

In [ ]:
# I AM THE LIVING MAN 2
for episode in trange(num_episodes):
    # Collect rollout samples
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)

        message1_policy = message1_actor.get_normal_policy(state)
        message1 = message1_policy.sample()
        message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)

        message2_policy = message2_actor.get_normal_policy(torch.zeros_like(state), message1)
        message2 = message2_policy.sample()
        message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)

        final_policy = final_actor.get_normal_policy(message2)
        action = final_policy.sample()
        action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)

        final_reward = reward_fn(state, action)

    # Update critics and actors
    for i in range(k):
        mb_state           = state[i * batch_size:(i + 1) * batch_size]
        mb_message1        = message1[i * batch_size:(i + 1) * batch_size]
        mb_message1_lp     = message1_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_message2        = message2[i * batch_size:(i + 1) * batch_size]
        mb_message2_lp     = message2_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action          = action[i * batch_size:(i + 1) * batch_size]
        mb_action_lp       = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward    = final_reward[i * batch_size:(i + 1) * batch_size]

        # ── Critics ──────────────────────────────────────────────────────────
        message1_critic_opt.zero_grad()
        message1_critic_value = message1_critic(mb_state)                        # V(s)
        message2_critic_value = message2_critic(mb_state, mb_message1)           # V(s, m1)
        final_critic_value    = final_critic(mb_state, mb_message1, mb_message2) # V(s, m1, m2)

        message1_critic_loss = mse_loss(message1_critic_value, message2_critic_value.detach())
        message1_critic_loss.backward()
        message1_critic_opt.step()

        message2_critic_opt.zero_grad()
        message2_critic_loss = mse_loss(message2_critic_value, final_critic_value.detach())
        message2_critic_loss.backward()
        message2_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_loss = mse_loss(final_critic_value, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # ── Marginal advantages ───────────────────────────────────────────────
        # Recompute with fresh critic values (post-update)
        with torch.no_grad():
            v0 = message1_critic(mb_state)
            v1 = message2_critic(mb_state, mb_message1)
            v2 = final_critic(mb_state, mb_message1, mb_message2)

        message1_advantage = v1 - v0                        # what did m1 add
        message2_advantage = v2 - v1                        # what did m2 add
        final_advantage    = mb_final_reward - v2           # what did action add

        def normalize(adv):
            return (adv - adv.mean()) / (adv.std() + 1e-8)

        message1_advantage = normalize(message1_advantage)
        message2_advantage = normalize(message2_advantage)
        final_advantage    = normalize(final_advantage)

        # ── Actor 1 ───────────────────────────────────────────────────────────
        message1_actor_opt.zero_grad()
        message1_policy       = message1_actor.get_normal_policy(mb_state)
        message1_new_lp       = message1_policy.log_prob(mb_message1).sum(dim=-1, keepdim=True)
        message1_ratio        = (message1_new_lp - mb_message1_lp).exp()
        message1_ratio_clip   = message1_ratio.clamp(1 - epsilon, 1 + epsilon)
        message1_ent          = message1_policy.entropy().mean()
        message1_actor_loss   = -torch.min(
            message1_ratio * message1_advantage,
            message1_ratio_clip * message1_advantage
        ).mean() - alpha * message1_ent
        message1_actor_loss.backward()
        message1_actor_opt.step()

        # ── Actor 2 ───────────────────────────────────────────────────────────
        message2_actor_opt.zero_grad()
        message2_policy       = message2_actor.get_normal_policy(torch.zeros_like(mb_state), mb_message1)
        message2_new_lp       = message2_policy.log_prob(mb_message2).sum(dim=-1, keepdim=True)
        message2_ratio        = (message2_new_lp - mb_message2_lp).exp()
        message2_ratio_clip   = message2_ratio.clamp(1 - epsilon, 1 + epsilon)
        message2_ent          = message2_policy.entropy().mean()
        message2_actor_loss   = -torch.min(
            message2_ratio * message2_advantage,
            message2_ratio_clip * message2_advantage
        ).mean() - alpha * message2_ent
        message2_actor_loss.backward()
        message2_actor_opt.step()

        # ── Final actor ───────────────────────────────────────────────────────
        final_actor_opt.zero_grad()
        final_policy          = final_actor.get_normal_policy(mb_message2)
        final_new_lp          = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
        final_ratio           = (final_new_lp - mb_action_lp).exp()
        final_ratio_clip      = final_ratio.clamp(1 - epsilon, 1 + epsilon)
        final_ent             = final_policy.entropy().mean()
        final_actor_loss      = -torch.min(
            final_ratio * final_advantage,
            final_ratio_clip * final_advantage
        ).mean() - alpha * final_ent
        final_actor_loss.backward()
        final_actor_opt.step()

    if episode % 100 == 0:
        print(
            f"{episode} | "
            f"fcl: {final_critic_loss.item():.4f} | "
            f"m2cl: {message2_critic_loss.item():.4f} | "
            f"m1cl: {message1_critic_loss.item():.4f} | "
            f"fr: {final_reward.mean().item():.4f}"
        )